# GNNHAR on S&P 500 + HOSE (Colab A100)

Faithful GNNHAR (Zhang, Pu, Cucuringu & Dong, IJF 2024, arXiv:2308.01419) under our full-matrix protocol on **both** markets: daily Parkinson variance, horizons {1,5,10,22}, QLIKE + date-clustered Diebold-Mariano. Reported models, paper-baseline-first: **HAR, GHAR** (paper baselines) -> **GBM, GBM+corr** (our own-history gamma-GBM) -> **GNNHAR\*** learned GNN variants (corr / no-graph control / random-edge placebo).

Each run writes ONE JSON per horizon (`results/gnnhar/gnnhar_<market>_h<h>.json`) carrying test/train/val metrics + a `fit_diagnostics` over/under-fit verdict per learned GNN; HOSE additionally carries `per_fold_qlike` (COVID-2020 / 2022 / Apr-2025 regime spikes dominate HOSE QLIKE and must be visible).

Workflow: git-clone CODE from master, Drive-mount DATA, run SMOKE=True first then SMOKE=False, display the overfit report, then commit the (gate-compliant) result JSONs to GitHub. Set runtime to **A100 GPU**.

In [ ]:
# 0. GPU (want A100)
get_ipython().system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

In [ ]:
# 1. Clone CODE from master (driver + full_matrix + vn_gbm_graph_stage1 + deps).
REPO_URL = 'https://github.com/ntquy9901/stock_vol_prediction01.git'
import os, shutil
if os.path.isdir('/content/repo'):
    shutil.rmtree('/content/repo')
get_ipython().system(f'git clone --depth 1 {REPO_URL} /content/repo')
for p in ('scripts/eda/gnnhar_sp500.py', 'scripts/eda/full_matrix.py', 'scripts/eda/vn_gbm_graph_stage1.py',
          'scripts/quality_gate/overfit_check.py', 'results/gamma_gbm/sp500_sectors.json'):
    print(('HAS ' if os.path.exists('/content/repo/' + p) else 'MISSING ') + p)

In [ ]:
# 2. DATA from Google Drive. sp500_clean = Yahoo (redistribution-restricted); HOSE enriched is also local-only.
#    Both are pulled from the Drive bundle (data/processed_enriched/*). Tries known folder layouts.
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = ['/content/drive/MyDrive/luanvan_data/colab_bundle_sp500_clean.zip',
              '/content/drive/MyDrive/public_bk/luanvan_data/colab_bundle_sp500_clean.zip']
DRIVE_ZIP = next((c for c in CANDIDATES if os.path.exists(c)), None)
assert DRIVE_ZIP, 'bundle not found; set DRIVE_ZIP manually. Tried: ' + str(CANDIDATES)
print('using', DRIVE_ZIP)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    members = [m for m in z.namelist() if m.startswith('data/processed_enriched/')]
    z.extractall('/content/repo', members)
print('unpacked', len(members), 'data files')
get_ipython().system('ls /content/repo/data/processed_enriched')

In [ ]:
# 3. Deps. Colab ships torch+CUDA; the driver is pure torch (no torch_geometric).
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
get_ipython().system('pip -q install pandas numpy scikit-learn pyarrow')

In [ ]:
# 4. Run GNNHAR (GPU) on BOTH markets. SMOKE=True first (1 fold, few epochs) to verify fast; then SMOKE=False.
import os, subprocess, time
SMOKE = True
MARKETS = [m for m in ('sp500', 'hose')
           if os.path.isdir('/content/repo/data/processed_enriched/' + ('sp500_clean' if m == 'sp500' else m))]
print('markets with data:', MARKETS)
assert MARKETS, 'no market data unpacked -- check the Drive bundle'
for MK in MARKETS:
    print('\n########## ' + MK + ' (smoke=' + str(SMOKE) + ') ##########', flush=True)
    cmd = ['python', 'scripts/eda/gnnhar_sp500.py', '--market', MK] + (['--smoke'] if SMOKE else [])
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd='/content/repo', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    print('[' + MK + '] exit=' + str(p.returncode) + '  ' + format((time.time() - t0) / 60, '.1f') + ' min')

In [ ]:
# 5. Inspect QLIKE + DISPLAY the over/under-fit report (fit_diagnostics) per learned GNN; HOSE per-fold QLIKE.
import glob, json
res = sorted(glob.glob('/content/repo/results/gnnhar/*.json'))
print('results:', [r.split('/')[-1] for r in res])
assert res, 'no result JSON -- did cell 4 finish?'
for f in res:
    d = json.load(open(f))
    print('\n== ' + f.split('/')[-1] + ' (n=' + str(d.get('n')) + ', folds=' + str(d.get('n_folds')) + ') ==')
    for m in d.get('metrics', {}):
        print('  ' + m.ljust(20) + ' QLIKE ' + format(d['metrics'][m]['qlike'], '.4f'))
    print('  -- fit_diagnostics (learned GNNs; overfit/underfit = real finding, not masked) --')
    for m, v in d.get('fit_diagnostics', {}).items():
        print('  ' + m.ljust(20) + ' ' + v.get('status', '?') + '  ' + str(v.get('reasons', [])))
    if 'per_fold_qlike' in d:
        print('  -- per_fold_qlike (HOSE regime-spike visibility) --')
        for m, arr in d['per_fold_qlike'].items():
            print('  ' + m.ljust(20) + ' ' + str([round(x, 4) for x in arr]))

In [ ]:
# 6. Commit the (gate-compliant) FULL result JSONs to GitHub. Run this only after SMOKE=False finished.
#    The JSONs carry train/val/test fit evidence, so the local overfit-evidence pre-push gate accepts them.
import glob, subprocess
res = [r for r in sorted(glob.glob('/content/repo/results/gnnhar/*_h*.json')) if '_smoke_' not in r]
print('full result JSONs to push:', [r.split('/')[-1] for r in res])
assert res, 'no FULL result JSONs -- set SMOKE=False in cell 4 and rerun before pushing'
from getpass import getpass
GH_USER = 'ntquy9901'
GH_TOKEN = getpass('GitHub token (fine-grained, Contents:write on this repo): ')
def sh(cmd, show=True):
    p = subprocess.run(cmd, cwd='/content/repo', shell=True, text=True, capture_output=True)
    if show:
        print(p.stdout, p.stderr)
    return p.returncode
sh('git config user.email "colab-a100@local" && git config user.name "colab-a100"')
sh('git add -f results/gnnhar/gnnhar_sp500_h*.json results/gnnhar/gnnhar_hose_h*.json')
sh('git commit -m "gnnhar sp500+hose results with over/under-fit evidence (Colab A100)"')
PUSH = 'git push https://' + GH_USER + ':' + GH_TOKEN + '@github.com/' + GH_USER + '/stock_vol_prediction01.git HEAD:master'
rc = sh(PUSH, show=False)
if rc != 0:
    print('push rejected -> fetch + rebase + retry')
    sh('git fetch origin master && git rebase origin/master')
    rc = sh(PUSH, show=False)
print('push OK' if rc == 0 else 'push FAILED -- check token/permissions')
del GH_TOKEN, PUSH
print('done -- locally: git pull origin master')

**Over/under-fit report:** cell 5 prints `fit_diagnostics` per learned GNN. An `overfit`/`underfit` verdict is a REAL finding (kept, not masked). On a single-fold SMOKE run HOSE GNNs can read `overfit` because the one smoke test window sits on a regime spike (2022 crash) whose val->test QLIKE gap exceeds the 25% threshold; the full multi-fold run pools across regimes and `per_fold_qlike` exposes which fold dominates.

**Token note:** use a fine-grained GitHub token scoped to this repo (Contents: Read and write); `getpass` keeps it out of the notebook and the cell deletes it after the push. The result JSONs carry the fit evidence so they pass the local overfit-evidence gate; keep local pushes paused while a Colab push is in flight (the cell fetches+rebases+retries if rejected).